In [49]:
"""
@
Auteurs:        Jeffrey Jason Boekstaaf, Tim Paulus van Croimvort en Haydar Eryörük
Studentnummers: 500460365, 500916516 en 500910901
Datum:          23-04-2026
Vak:            Beroepsproject 3.4
Opleiding:      Toegepaste Wiskunde & Data Science
School:         Hogeschool van Amsterdam
"""

import csv
import os
import pandas as pd
import pickle

from pathlib import Path
from sklearn.pipeline import Pipeline

# Hier wordt de data ingeladen waarop de winstverwachting wordt uitgerekend.
data_ev = pd.read_csv("churndata_deployment.csv")


In [50]:
# Haalt de Gradient Boosting classifier op uit de lokale map.
with open('pipelineGB.pkl', 'rb') as file:
    gradient_boosting_pipe = pickle.load(file)


In [51]:
# Uit de gridsearch is dit aantal van n estimators (bomen) als beste uitgekomen en daarom dient het hier aangepast te worden.
gradient_boosting_pipe.n_estimators = 3200


In [52]:
# Hier wordt de voorspelling gedaan op de data wie wel of niet zouden willen overstappen.
predicted = gradient_boosting_pipe.predict(data_ev)


In [53]:
if os.path.exists("tarief2.csv"):
    os.remove("tarief2.csv")


In [54]:
rows = []

for i in range(len(predicted)):
    if (predicted[i]):
        rows.append([(i + 1), "Ja"])
    else:
        rows.append([(i + 1), "Nee"])

df = pd.DataFrame(data = rows, columns = ["Klantnummer", "Beslissing"])


In [55]:
df.to_excel("tarief2.xlsx", index = False)
file = Path("tarief2.xlsx")
file.rename(file.with_suffix(".csv"))


WindowsPath('tarief2.csv')

In [56]:
# Hier wordt de winstverwachting uitgerekend en getoond op de klanten die zouden willen overstappen.
W = (((data_ev['Seconds of Use'] / 60.0) * 0.2) + (data_ev['Frequency of SMS'] * 0.07))
expected_profit = (1 - (0.25 * predicted)) * W
print("De winstverwachting is: €", (round(expected_profit.sum(), 2)), sep = "", end = "0.")


De winstverwachting is: €6838.30.